In [ ]:
# This notebook is about how to build sequential workflow using langgraph

In [87]:
#import

from langgraph.graph import StateGraph, START, END
from typing import TypedDict 

from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

In [88]:
load_dotenv 

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

In [89]:

model = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    temperature=0
)

In [90]:
# define a state
class BlogState(TypedDict):
    content: str
    topic: str
    outline: str
    rating: int


In [91]:
def create_outline(state: BlogState) -> BlogState:
    # Fetch Topic
    topic = state['topic']

    #call llm gen outline
    prompt = f'generate a detailed outline for a blog on the topic- {topic}'
    outline = model.invoke(prompt).content

    #update state
    state['outline'] = outline

    return state

In [92]:
def create_blog(state: BlogState) -> BlogState:
    topic = state['topic']
    outline = state['outline']

    prompt = f'write a detailed on the topic - {topic} using the following outline \n {outline}'

    content = model.invoke(prompt).content
    state['content']= content

    return state

In [93]:
def evaluate(state: BlogState) -> BlogState:
    #based on the outline rate the blog

    #fetch
    outline = state['outline']
    content = state['content']

    prompt = f'based on the outline- {outline} rate the following blog on the scale of 1 to 10  \n {content}'

    # call llm
    rating = model.invoke(prompt)

    state['rating'] = rating.content


    return state


    

In [94]:
# define a graph
graph = StateGraph(BlogState)

# add nodes to the graph
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)
graph.add_node('evaluate', evaluate)

# add edges to the graph
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate')
graph.add_edge('evaluate', END)

# compile the graph
workflow = graph.compile()



In [95]:
#execute the graph
initial_state = {'topic': 'the importance of the water in human body'}
final_state = workflow.invoke(initial_state)
print(final_state)

{'content': [{'type': 'text', 'text': '# The Elixir of Life: Why Water is Vital for Human Health\n\nImagine your body as a high-performance engine. You can put in the best fuel, maintain the parts, and polish the exterior, but without the ultimate lubricant and coolant, the entire system grinds to a halt. In the human body, that indispensable substance is water.\n\nIt is often said that water is life, and the science backs this up completely. Water makes up roughly **50% to 70% of the adult human body**, depending on age, gender, and body composition. Infants can be up to 75% water, while older adults hover closer to 50%. \n\nDespite its critical importance, millions of people live in a state of mild, chronic dehydration without even realizing it. They attribute their low energy, frequent headaches, and sluggish digestion to stress or aging, when the real culprit is simply an empty water glass. \n\nWater is not just a beverage to quench your thirst; it is a fundamental fuel that drives

In [96]:
print(final_state['outline'])

[{'type': 'text', 'text': 'Here is a comprehensive, SEO-friendly, and detailed blog post outline on **"The Importance of Water in the Human Body."**\n\n---\n\n# Blog Post Outline: The Elixir of Life: Why Water is Vital for Human Health\n\n## Working Blog Titles (SEO Options):\n* *The Elixir of Life: Why Water is Essential for Your Health*\n* *H2O and You: The Science-Backed Importance of Staying Hydrated*\n* *More Than Thirst: 7 Vital Functions of Water in the Human Body*\n* *What Happens to Your Body When You Drink Enough Water?*\n\n---\n\n## I. Introduction\n* **The Hook:** Start with a compelling metaphor (e.g., comparing the human body to a high-performance engine that needs the ultimate lubricant to run).\n* **The Surprising Statistic:** Highlight that the human body is made up of **50% to 70% water** (depending on age, gender, and composition).\n* **Problem Statement:** Most people live in a state of mild, chronic dehydration without even realizing it.\n* **Thesis Statement:** Wa

In [97]:
print(final_state['content'])

[{'type': 'text', 'text': '# The Elixir of Life: Why Water is Vital for Human Health\n\nImagine your body as a high-performance engine. You can put in the best fuel, maintain the parts, and polish the exterior, but without the ultimate lubricant and coolant, the entire system grinds to a halt. In the human body, that indispensable substance is water.\n\nIt is often said that water is life, and the science backs this up completely. Water makes up roughly **50% to 70% of the adult human body**, depending on age, gender, and body composition. Infants can be up to 75% water, while older adults hover closer to 50%. \n\nDespite its critical importance, millions of people live in a state of mild, chronic dehydration without even realizing it. They attribute their low energy, frequent headaches, and sluggish digestion to stress or aging, when the real culprit is simply an empty water glass. \n\nWater is not just a beverage to quench your thirst; it is a fundamental fuel that drives every singl

In [98]:
print(final_state['rating'])

[{'type': 'text', 'text': '**Rating: 9.5 / 10**\n\n### **Detailed Evaluation**\n\n#### **1. Structure & Adherence to Outline (10/10)**\n* **Perfect Alignment:** The blog post strictly follows every section, subsection, and bullet point specified in the outline. \n* **Seamless Flow:** The transitions between the introductory engine metaphor, anatomical science, health benefits, dehydration risks, myth-busting, and actionable "hacks" feel natural and cohesive.\n\n#### **2. Content & Completeness (9.5/10)**\n* **Accurate Scientific Data:** All body percentage statistics (e.g., brain/heart ~73%, lungs ~83%) and cellular concepts (evaporative cooling, cartilage composition, fluid-to-weight loss ratios) were fully expanded and explained clearly.\n* **Practical & Actionable:** The inclusion of "Habit Stacking," "Eat Your Water," and "The Urine Check" translates theoretical information into easy, practical advice for the reader.\n* **Nuance:** It appropriately included key nuances, such as cla